### Build and validate RAG ChromaDB

1. Update `EMBEDDING_PROVIDER` and `EMBEDDING_MODEL` trong file `.env`.
2. Dung Backend va xoa directory duoc cau hinh o `PERSIST_DIRECTORY` khi doi provider/model embedding.
3. Restart notebook kernel.
4. Khi build, uncomment/comment local or Google.
5. Xac nhan output `PASS` & `SUITABLE`.

In [1]:
import hashlib
import os
import sys
import time
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path("D:/KLTN/Project/BE_ChatBot")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)

from src.pipeline.rag_pipline import (
    RAGStorage,
    load_json_places,
    split_documents,
)

storage = RAGStorage()
print(f"Provider: {storage.embedding_provider.value}")
print(f"Model   : {storage.embedding_model_name}")
print(f"Database: {storage.persist_directory}")

D:\KLTN\Project\BE_ChatBot\src\pipeline\rag_pipline.py:152: SyntaxWarning: invalid escape sequence '\K'
  """


Embedding provider: google
Embedding model   : models/gemini-embedding-001
Provider: google
Model   : models/gemini-embedding-001
Database: src/db/chroma_db


In [2]:
started_at = time.perf_counter()

# Local HuggingFace/Ollama: uncomment this line and comment the Google line.
# vectorstore = storage.build_vector_db()

# Google API: uncomment this line and comment the local line.
vectorstore = storage.build_vector_db_google()

embedding_seconds = time.perf_counter() - started_at

📂 Tìm thấy 6 file JSON: ['dalat_100_tourist_spots.json', 'dn_100_tourist_spots.json', 'hcm_100_tourist_spots.json', 'hn_100_tourist_spots.json', 'nt_100_tourist_spots.json', 'vt_100_tourist_spots.json']
✅ Load xong: 600 địa điểm từ 6 file
Splitting documents into chunks...
Split into 600 chunks

[Chunk 1] DL_0001 | 364 chars
Khu du lịch Thác Datanla là một địa điểm thuộc Đà Lạt. Loại hình: waterfall. Các hoạt động nổi bật: ...

[Chunk 2] DL_0002 | 322 chars
Đồi chè Cầu Đất là một địa điểm thuộc Đà Lạt. Loại hình: nature. Các hoạt động nổi bật: săn mây,đồi ...

[Chunk 3] DL_0003 | 332 chars
Hồ Tuyền Lâm là một địa điểm thuộc Đà Lạt. Loại hình: lake. Các hoạt động nổi bật: thiên nhiên,chèo ...
Google batch 1: OK (100 chunks)
Waiting 65s before batch 2 to respect the Google free-tier rate limit...
Google batch 2: OK (100 chunks)
Waiting 65s before batch 3 to respect the Google free-tier rate limit...
Google batch 3: OK (100 chunks)
Waiting 65s before batch 4 to respect the Google free-tie

In [ ]:
json_data_dir = Path(os.getenv("JSON_DATA_DIR", "src/source_data/places_data"))
if not json_data_dir.is_absolute():
    json_data_dir = PROJECT_ROOT / json_data_dir

documents = load_json_places(str(json_data_dir))
chunks = split_documents(documents, chunk_size=1000, chunk_overlap=150)
stored = vectorstore.get(include=["metadatas", "documents"])
sample = vectorstore.get(limit=1, include=["embeddings"])

stored_ids = stored.get("ids") or []
stored_metadatas = stored.get("metadatas") or []
stored_contents = stored.get("documents") or []
embeddings = sample.get("embeddings")
if embeddings is None or len(embeddings) == 0:
    raise AssertionError("ChromaDB did not return a stored embedding.")

document_ids = [doc.metadata.get("place_id") for doc in documents]
chunk_ids = [chunk.metadata.get("place_id") for chunk in chunks]
stored_place_ids = [meta.get("place_id") for meta in stored_metadatas]
document_id_set = {place_id for place_id in document_ids if place_id}
chunk_id_set = {place_id for place_id in chunk_ids if place_id}
stored_id_set = {place_id for place_id in stored_place_ids if place_id}

expected_rows = sorted(
    f"{chunk.metadata.get('place_id', '')}\0{chunk.page_content}"
    for chunk in chunks
)
actual_rows = sorted(
    f"{metadata.get('place_id', '')}\0{content}"
    for metadata, content in zip(stored_metadatas, stored_contents)
)
expected_checksum = hashlib.sha256("\n".join(expected_rows).encode("utf-8")).hexdigest()
actual_checksum = hashlib.sha256("\n".join(actual_rows).encode("utf-8")).hexdigest()

document_regions = Counter(doc.metadata.get("region", "UNKNOWN") for doc in documents)
chunk_regions = Counter(chunk.metadata.get("region", "UNKNOWN") for chunk in chunks)
stored_regions = Counter(meta.get("region", "UNKNOWN") for meta in stored_metadatas)

print("\nCoverage by region:")
print(f"{'Region':<20} {'Documents':>10} {'Chunks':>10} {'Stored':>10}")
for region in sorted(set(document_regions) | set(chunk_regions) | set(stored_regions)):
    print(
        f"{region:<20} {document_regions[region]:>10} "
        f"{chunk_regions[region]:>10} {stored_regions[region]:>10}"
    )

missing_ids = chunk_id_set - stored_id_set
unexpected_ids = stored_id_set - chunk_id_set
missing_place_ids = sum(not place_id for place_id in chunk_ids + stored_place_ids)
empty_content = sum(not chunk.page_content.strip() for chunk in chunks) + sum(
    not (content or "").strip() for content in stored_contents
)
vector_dimension = len(embeddings[0])

checks = {
    "stored_count": len(stored_ids) == len(chunks),
    "document_ids_reach_chunks": document_id_set == chunk_id_set,
    "stored_place_ids": chunk_id_set == stored_id_set,
    "metadata_complete": missing_place_ids == 0,
    "content_complete": empty_content == 0 and expected_checksum == actual_checksum,
    "region_coverage": chunk_regions == stored_regions,
}
if (
    storage.embedding_provider.value == "google"
    and storage.embedding_model_name in {"gemini-embedding-001", "models/gemini-embedding-001"}
):
    checks["google_vector_dimension"] = vector_dimension == 3072

print("\n" + "-" * 60)
print(f"JSON files             : {len(list(json_data_dir.glob('*.json')))}")
print(f"Documents              : {len(documents)}")
print(f"Unique document IDs    : {len(document_id_set)}")
print(f"Chunks expected/stored : {len(chunks)}/{len(stored_ids)}")
print(f"Unique stored IDs      : {len(stored_id_set)}")
print(f"Missing IDs            : {len(missing_ids)}")
print(f"Unexpected IDs         : {len(unexpected_ids)}")
print(f"Missing place_id       : {missing_place_ids}")
print(f"Empty content          : {empty_content}")
print(f"Content checksum       : {actual_checksum}")
print(f"Checksum matches       : {'YES' if expected_checksum == actual_checksum else 'NO'}")
print(f"Vector dimension       : {vector_dimension}")
print(f"Embedding time         : {embedding_seconds:.2f}s")
print(f"Collection metadata    : {vectorstore._collection.metadata}")

for check_name, passed in checks.items():
    print(f"Check {check_name:<26}: {'PASS' if passed else 'FAIL'}")

if not all(checks.values()):
    print("NOT SUITABLE: data completeness verification failed.")
    raise AssertionError("Data completeness verification failed.")

print("SUITABLE: all chunks were embedded and stored successfully.")

📂 Tìm thấy 6 file JSON: ['dalat_100_tourist_spots.json', 'dn_100_tourist_spots.json', 'hcm_100_tourist_spots.json', 'hn_100_tourist_spots.json', 'nt_100_tourist_spots.json', 'vt_100_tourist_spots.json']
✅ Load xong: 600 địa điểm từ 6 file
Splitting documents into chunks...
Split into 600 chunks

[Chunk 1] DL_0001 | 364 chars
Khu du lịch Thác Datanla là một địa điểm thuộc Đà Lạt. Loại hình: waterfall. Các hoạt động nổi bật: ...

[Chunk 2] DL_0002 | 322 chars
Đồi chè Cầu Đất là một địa điểm thuộc Đà Lạt. Loại hình: nature. Các hoạt động nổi bật: săn mây,đồi ...

[Chunk 3] DL_0003 | 332 chars
Hồ Tuyền Lâm là một địa điểm thuộc Đà Lạt. Loại hình: lake. Các hoạt động nổi bật: thiên nhiên,chèo ...

Coverage by region:
Region                Documents     Chunks     Stored
Hà Nội                      100        100        100
Hồ Chí Minh                 100        100        100
Nha Trang                   100        100        100
Vũng Tàu                    100        100        100
Đà Lạt  

: 